# Model Merging and GGUF Conversion


This notebook runs the full post–fine-tuning pipeline to make a LoRA-fine-tuned model usable for lightweight inference with llama.cpp.

First, it loads the original full-precision baseline model and attaches the trained LoRA adapter produced during fine-tuning. Although the adapters may have been trained using a 4-bit (QLoRA) setup for efficiency, the merge is performed against the full-precision base model to produce a clean and standard standalone checkpoint.

Next, the notebook merges the LoRA weights into the baseline model and removes all adapter layers (merge_and_unload()), producing a single merged model that no longer depends on PEFT/LoRA at inference time. The merged model is then saved in Hugging Face format (FP16) as a reusable intermediate artifact.

Finally, the notebook converts the merged Hugging Face model to the GGUF format using llama.cpp, enabling portable inference outside the Python/PyTorch ecosystem. Two GGUF variants are generated:

FP16 GGUF (16-bit) for maximum fidelity and highest-quality inference.

Q8_0 GGUF (8-bit) for a smaller, easier-to-transfer model with minimal practical quality loss.

By the end, this notebook produces both full-precision and quantized GGUF files that can be used directly with llama-cli / llama-server for deployment and inference.

In [1]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from unsloth import FastLanguageModel


# Where your saved LoRA adapter folder is (the one you downloaded & re-uploaded / copied)
ADAPTER_DIR = Path("/workspace/MentorApp/lora_adapter").resolve()

# Where to save the merged Hugging Face model (this folder will be large)
MERGED_DIR = Path("/workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16").resolve()
MERGED_DIR.mkdir(parents=True, exist_ok=True)

LLAMA_DIR  = Path("/workspace/MentorApp/llama.cpp").resolve()

print("ADAPTER_DIR exists:", ADAPTER_DIR.exists(), ADAPTER_DIR)
print("MERGED_DIR:", MERGED_DIR)


/workspace/MentorApp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/tmp/ipykernel_10083/805959802.py:5: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
ADAPTER_DIR exists: True /workspace/MentorApp/lora_adapter
MERGED_DIR: /workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16


In [2]:
# Loading the baseline model
BASE_MODEL_FULL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Use FP16 for a clean FP16 -> GGUF pipeline (bfloat16 is also fine, but FP16 is the usual target)
dtype = torch.float16

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_FULL, use_fast=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_FULL,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]


In [3]:
# Attach LoRA adapter onto the full-precision base model
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))

# Merge LoRA weights into base and remove adapter layers
merged_model = model.merge_and_unload()

# (Optional) ensure merged weights are saved as FP16
merged_model = merged_model.to(dtype)

# Save merged HF model + tokenizer
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))

print("✅ Merged HF model saved to:", MERGED_DIR)
print("   Base used:", BASE_MODEL_FULL)

✅ Merged HF model saved to: /workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16
   Base used: Qwen/Qwen2.5-Coder-7B-Instruct


## Build llama.cpp and convert the HF --> GGUF 

In [4]:
import os, subprocess
from pathlib import Path

def run(cmd):
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True)

LLAMA_DIR = Path("/workspace/MentorApp/llama.cpp").resolve()

if not LLAMA_DIR.exists():
    run(f"git clone https://github.com/ggml-org/llama.cpp {LLAMA_DIR}")

# Clean build (release)
run(f"cd {LLAMA_DIR} && rm -rf build && cmake -B build -DCMAKE_BUILD_TYPE=Release")
run(f"cd {LLAMA_DIR} && cmake --build build -j")

# Python requirements for conversion
run(f"python -m pip install -U -r {LLAMA_DIR}/requirements.txt")

print("✅ llama.cpp ready at:", LLAMA_DIR)


>> git clone https://github.com/ggml-org/llama.cpp /workspace/MentorApp/llama.cpp


Cloning into '/workspace/MentorApp/llama.cpp'...


>> cd /workspace/MentorApp/llama.cpp && rm -rf build && cmake -B build -DCMAKE_BUILD_TYPE=Release
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.43.0") 


CMAKE_BUILD_TYPE=Release


-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5") 
-- Found OpenMP_CXX: -fopenmp (found version "4.5") 
-- Found OpenMP: TRUE (found version "4.5")  
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.5
-- ggml commit:  292f6908c
-- Found OpenSSL: /usr/lib/x86_64-linux-gnu/libcrypto.so (found version "3.0.13")  
-- Performing Test OPENSSL_VERSION_SUPPORTED
-- Performing Test OPENSSL_VERSION_SUPPORTED - Success
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: common
-- Configuring done (2.2s)
-- Generating done (0.3s)
-- Build fi

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xformers 0.0.34 requires torch==2.10.0, but you have torch 2.6.0+cpu which is incompatible.
torchvision 0.25.0 requires torch==2.10.0, but you have torch 2.6.0+cpu which is incompatible.


✅ llama.cpp ready at: /workspace/MentorApp/llama.cpp


In [5]:
def run(cmd):
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True)

LLAMA_DIR  = Path("/workspace/MentorApp/llama.cpp").resolve()

# Your merged HF model folder (this is the one you just created)
MERGED_DIR = Path("/workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16").resolve()

# Save GGUF files in a dedicated folder
OUT_DIR    = Path("/workspace/MentorApp/outputs/GGUF").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

GGUF_F16 = OUT_DIR / "qwen2_5_coder_7b_merged_f16.gguf"
GGUF_Q8  = OUT_DIR / "qwen2_5_coder_7b_merged_q8_0.gguf"

print("MERGED_DIR exists:", MERGED_DIR.exists(), MERGED_DIR)
print("OUT_DIR:", OUT_DIR)
print("GGUF_F16:", GGUF_F16)
print("GGUF_Q8 :", GGUF_Q8)

# 1) Convert merged HF -> GGUF (FP16)  [required as input to quantization]
run(
    f"python {LLAMA_DIR}/convert_hf_to_gguf.py {MERGED_DIR} "
    f"--outfile {GGUF_F16} --outtype f16"
)

print("✅ GGUF (F16) saved to:", GGUF_F16)

# 2) Quantize FP16 GGUF -> 8-bit GGUF (Q8_0)
run(
    f"{LLAMA_DIR}/build/bin/llama-quantize "
    f"{GGUF_F16} {GGUF_Q8} Q8_0"
)

print("✅ GGUF (Q8_0) saved to:", GGUF_Q8)

MERGED_DIR exists: True /workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16
OUT_DIR: /workspace/MentorApp/outputs/GGUF
GGUF_F16: /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf
GGUF_Q8 : /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_q8_0.gguf
>> python /workspace/MentorApp/llama.cpp/convert_hf_to_gguf.py /workspace/MentorApp/outputs/merged_hf_qwen2_5_instruct_7b_fp16 --outfile /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf --outtype f16


INFO:hf-to-gguf:Loading model: merged_hf_qwen2_5_instruct_7b_fp16
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {3584, 152064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {18944, 3584}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {3584, 18944}
INFO:hf-to-gguf:blk.0.ffn_up.weight,    

✅ GGUF (F16) saved to: /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf
>> /workspace/MentorApp/llama.cpp/build/bin/llama-quantize /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_q8_0.gguf Q8_0


main: build = 7978 (292f6908c)
main: built with GNU 13.3.0 for Linux x86_64
main: quantizing '/workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf' to '/workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 27 key-value pairs and 339 tensors from /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              =


main: quantize time = 28260.07 ms
main:    total time = 28260.07 ms
✅ GGUF (Q8_0) saved to: /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_q8_0.gguf
